# Lab 6 — Single-node HDFS

## 1. Setup

This notebook runs on its own: the cells below recreate what it uses from notebook 01 (Spark session, `corpus.txt`, `mapper.py`, `reducer.py` and `orders.csv`).

In [1]:
# !pip install -q pyspark==4.0.1 pyarrow

In [2]:
import os, re, subprocess, shutil, time, random, glob
from collections import defaultdict
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Lab 6 - Hadoop")
    .master("local[*]")
    # spark.hadoop.* goes straight to the Hadoop configuration. Here: 4 MB blocks on the local disk
    # (the default is 32 MB), so we can see several blocks with a small file
    .config("spark.hadoop.fs.local.block.size", str(4 * 2**20))
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

BASE = "/tmp/lab6"
os.makedirs(BASE, exist_ok=True)

def sh(cmd, check=True):
    """Runs a shell command and shows its output (equivalent to ! in a notebook)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

print("Spark            :", spark.version)
print("Hadoop (client)  :", spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:28:50 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:28:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 18:28:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark            : 4.0.1
Hadoop (client)  : 3.4.1


In [3]:
jvm  = spark._jvm
conf = spark._jsc.hadoopConfiguration()

for scheme in ["file", "hdfs", "webhdfs", "s3a", "gs", "abfs"]:
    try:
        impl = jvm.org.apache.hadoop.fs.FileSystem.getFileSystemClass(scheme, conf).getName()
        print(f"{scheme + '://':<11} -> {impl}")
    except Exception:
        print(f"{scheme + '://':<11} -> (connector not installed)")

file://     -> org.apache.hadoop.hive.ql.io.ProxyLocalFileSystem
hdfs://     -> org.apache.hadoop.hdfs.DistributedFileSystem
webhdfs://  -> org.apache.hadoop.hdfs.web.WebHdfsFileSystem
s3a://      -> (connector not installed)
gs://       -> (connector not installed)
abfs://     -> (connector not installed)


In [4]:
VOCAB = ("data spark hadoop hdfs yarn mapreduce cluster block file table python pandas "
         "parquet delta streaming window model pipeline shuffle partition memory disk network "
         "driver executor task stage job lake bronze silver gold").split()
weights = [1 / (i + 1) for i in range(len(VOCAB))]          # Zipf distribution: a few very frequent words

rng = random.Random(42)
CORPUS = f"{BASE}/corpus.txt"
with open(CORPUS, "w") as f:
    for _ in range(600_000):
        f.write(" ".join(rng.choices(VOCAB, weights, k=12)) + "\n")

path   = jvm.org.apache.hadoop.fs.Path(f"file://{CORPUS}")
fs     = path.getFileSystem(conf)
status = fs.getFileStatus(path)

print(f"Size             : {status.getLen() / 1e6:.1f} MB")
print(f"Block size       : {status.getBlockSize() / 2**20:.0f} MB  (set in the setup)")
print(f"Replication      : {status.getReplication()}")
print(f"Logical blocks   : {-(-status.getLen() // status.getBlockSize())}")

Size             : 45.0 MB
Block size       : 4 MB  (set in the setup)
Replication      : 1
Logical blocks   : 11


In [5]:
MAPPER = r'''
import sys, re
for line in sys.stdin:
    for word in re.findall(r"[a-z]+", line.lower()):
        print(f"{word}\t1")
'''

REDUCER = r'''
import sys
current, count = None, 0
for line in sys.stdin:
    word, n = line.rstrip("\n").split("\t")
    if word != current:              # new key: the previous one is done (the input is sorted)
        if current is not None:
            print(f"{current}\t{count}")
        current, count = word, 0
    count += int(n)
if current is not None:
    print(f"{current}\t{count}")
'''

with open(f"{BASE}/mapper.py", "w") as f:  f.write(MAPPER)
with open(f"{BASE}/reducer.py", "w") as f: f.write(REDUCER)

print("Mapper output (start):")
sh(f"head -n 1 {CORPUS} | python3 {BASE}/mapper.py")

Mapper output (start):
block	1
data	1
spark	1
data	1
python	1
file	1
memory	1
data	1
hadoop	1
data	1
data	1
hdfs	1


In [6]:
ORDERS_CSV = f"{BASE}/orders.csv"
rng = random.Random(7)
with open(ORDERS_CSV, "w") as f:
    f.write("order_id,region,price,quantity\n")
    for i in range(200_000):
        f.write(f"{i},{rng.choice(['North','Northeast','Midwest','Southeast','South'])},"
                f"{rng.uniform(10, 500):.2f},{rng.randint(1, 5)}\n")

---
## 2. Hands-on: single-node HDFS

Now we will install Hadoop and start HDFS in **pseudo-distributed mode**: NameNode and DataNode running on the same machine, as separate processes. It is the same software as a 1,000-node cluster, just with one node.

> This section was designed for Colab or Linux with Java installed. The download is about 500 MB and takes a few minutes. If the link is down, check the versions at https://archive.apache.org/dist/hadoop/common/

In [7]:
HADOOP_VERSION = "3.4.1"          # same version as the Hadoop client bundled with PySpark 4.0
HADOOP_HOME    = f"/opt/hadoop-{HADOOP_VERSION}"
MIRROR = f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VERSION}"
# "lean" version (no AWS SDK, ~half the size); if missing, downloads the full one
URLS = [f"{MIRROR}/hadoop-{HADOOP_VERSION}-lean.tar.gz", f"{MIRROR}/hadoop-{HADOOP_VERSION}.tar.gz"]

if not os.path.exists(HADOOP_HOME):
    sh(" || ".join(f"wget -q {u} -O /tmp/hadoop.tar.gz" for u in URLS))
    sh("tar -xzf /tmp/hadoop.tar.gz -C /opt")
    os.remove("/tmp/hadoop.tar.gz")

JAVA_HOME = os.path.dirname(os.path.dirname(os.path.realpath(shutil.which("java"))))
os.environ.update({
    "JAVA_HOME":       JAVA_HOME,
    "HADOOP_HOME":     HADOOP_HOME,
    "HADOOP_CONF_DIR": f"{HADOOP_HOME}/etc/hadoop",
    "PATH":            f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"],
    # Colab runs as root; Hadoop requires this to be declared
    "HDFS_NAMENODE_USER": "root", "HDFS_DATANODE_USER": "root", "HDFS_SECONDARYNAMENODE_USER": "root",
})
sh("hadoop version | head -n 1")
print("JAVA_HOME:", JAVA_HOME)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



RuntimeError: command failed (2): tar -xzf /tmp/hadoop.tar.gz -C /opt

### Configuration

Two XML files define the cluster:

| File | Property | Value | Meaning |
|---------|-------------|-------|-------------|
| `core-site.xml` | `fs.defaultFS` | `hdfs://localhost:9000` | NameNode address |
| `hdfs-site.xml` | `dfs.replication` | `1` | We only have one DataNode |
| `hdfs-site.xml` | `dfs.namenode.name.dir` | local folder | Where the NameNode stores metadata |
| `hdfs-site.xml` | `dfs.datanode.data.dir` | local folder | Where the DataNode stores blocks |

In [ ]:
HDFS_DATA = "/tmp/hdfs-data"

def write_site(name, props):
    body = "\n".join(f"  <property><name>{k}</name><value>{v}</value></property>" for k, v in props.items())
    with open(f"{HADOOP_HOME}/etc/hadoop/{name}", "w") as f:
        f.write(f"<?xml version=\"1.0\"?>\n<configuration>\n{body}\n</configuration>\n")

write_site("core-site.xml", {"fs.defaultFS": "hdfs://localhost:9000"})
write_site("hdfs-site.xml", {
    "dfs.replication":         1,
    "dfs.namenode.name.dir":   f"file://{HDFS_DATA}/namenode",
    "dfs.datanode.data.dir":   f"file://{HDFS_DATA}/datanode",
    "dfs.permissions.enabled": "false",
})

# Java 17+ (required by PySpark 4) must open some internal modules to Hadoop
with open(f"{HADOOP_HOME}/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write(f"\nexport JAVA_HOME={JAVA_HOME}\n")
    f.write('export HADOOP_OPTS="$HADOOP_OPTS --add-opens=java.base/java.lang=ALL-UNNAMED '
            '--add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED"\n')

sh(f"cat {HADOOP_HOME}/etc/hadoop/hdfs-site.xml")

In [ ]:
# Format the NameNode (first time only: it erases the metadata!) and start both processes
shutil.rmtree(HDFS_DATA, ignore_errors=True)
sh("hdfs namenode -format -force -nonInteractive 2>&1 | grep -i 'successfully formatted'", check=False)

sh("hdfs --daemon start namenode")
sh("hdfs --daemon start datanode")
sh("hdfs dfsadmin -safemode wait")     # the NameNode starts in read-only mode until the DataNodes register

sh("jps", check=False)                 # running Java processes
sh("hdfs dfsadmin -report | head -n 12")

`jps` should list `NameNode` and `DataNode`. On local Linux, the NameNode web interface is at http://localhost:9870 (on Colab it is not directly accessible).

### 2.1 `hdfs dfs` commands

The commands mimic Linux ones, with `hdfs dfs -` in front:

| Linux | HDFS |
|-------|------|
| `ls -lh` | `hdfs dfs -ls -h` |
| `mkdir -p` | `hdfs dfs -mkdir -p` |
| `cp local destination` | `hdfs dfs -put local destination` |
| `cp destination local` | `hdfs dfs -get destination local` |
| `cat` / `tail` | `hdfs dfs -cat` / `hdfs dfs -tail` |
| `du -sh` | `hdfs dfs -du -s -h` |
| `rm -r` | `hdfs dfs -rm -r` |

In [ ]:
sh("hdfs dfs -mkdir -p /lab6/raw")
sh(f"hdfs dfs -put -f {CORPUS} /lab6/raw/corpus.txt")

# block size is set per file, at write time: here, 4 MB to produce several blocks
sh(f"hdfs dfs -D dfs.blocksize={4 * 2**20} -put -f {CORPUS} /lab6/raw/corpus_4mb.txt")

sh("hdfs dfs -ls -h /lab6/raw")
sh("hdfs dfs -du -s -h /lab6")
sh("hdfs dfs -cat /lab6/raw/corpus.txt | head -n 2")

In [ ]:
# fsck shows a file's blocks and which DataNodes hold them
sh("hdfs fsck /lab6/raw/corpus_4mb.txt -files -blocks -locations | grep -E 'blk_|Total blocks|corpus' | head -n 8")

In [ ]:
# Blocks are just regular files on the DataNode's disk
sh(f"find {HDFS_DATA}/datanode -name 'blk_*' -not -name '*.meta' | head -n 5")
sh(f"find {HDFS_DATA}/datanode -name 'blk_*' -not -name '*.meta' | wc -l")

Each `blk_<id>` is a piece of up to 4 MB of the file, and the `.meta` next to it holds the checksums used to detect corruption. The NameNode only knows **which blocks make up each file and where they are**; the content never passes through it.

### 2.2 Spark reading and writing on HDFS

To Spark, HDFS is just another URL scheme. The code is the same as in the previous labs.

In [ ]:
HDFS = "hdfs://localhost:9000"

print("Partitions (textFile, 1 per block):", sc.textFile(f"{HDFS}/lab6/raw/corpus_4mb.txt").getNumPartitions())

words = (
    spark.read.text(f"{HDFS}/lab6/raw/corpus.txt")
    .select(F.explode(F.split("value", " ")).alias("word"))
    .groupBy("word").count()
)
words.write.mode("overwrite").parquet(f"{HDFS}/lab6/gold/wordcount")

sh("hdfs dfs -ls -h /lab6/gold/wordcount")
spark.read.parquet(f"{HDFS}/lab6/gold/wordcount").orderBy(F.desc("count")).show(5)

### 2.3 A real MapReduce job

The same `mapper.py` and `reducer.py` from section 4.2 of notebook 01, now run by Hadoop Streaming, reading from and writing to HDFS. With no YARN configured, Hadoop uses the local executor (`mapreduce.framework.name=local`), but the flow is identical to a cluster's.

In [ ]:
streaming_jar = glob.glob(f"{HADOOP_HOME}/share/hadoop/tools/lib/hadoop-streaming-*.jar")[0]
sh("hdfs dfs -rm -r -f /lab6/out/wordcount_mr")

t0 = time.perf_counter()
sh(f"""hadoop jar {streaming_jar} \
      -files {BASE}/mapper.py,{BASE}/reducer.py \
      -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
      -input /lab6/raw/corpus_4mb.txt -output /lab6/out/wordcount_mr 2>&1 | grep -E 'map 100%|completed successfully|Launched map tasks'""",
   check=False)
print(f"\nMapReduce job time: {time.perf_counter() - t0:.1f} s")

sh("hdfs dfs -ls /lab6/out/wordcount_mr")
sh("hdfs dfs -cat /lab6/out/wordcount_mr/part-* | sort -k2 -nr | head -n 5")

The `_SUCCESS` file is a marker: the job finished and the output is complete. Readers check for this file before consuming the folder. Spark writes the same marker, a direct inheritance from Hadoop.

### 2.4 Pandas and PyArrow reading HDFS

Pandas does not speak HDFS on its own. There are two ways:

| Way | How it works | Pros and cons |
|---------|---------------|----------------|
| **PyArrow `HadoopFileSystem`** | Uses the native `libhdfs` library (which starts a JVM inside Python) | Fast, reads Parquet directly; requires Hadoop installed and `CLASSPATH` configured |
| **WebHDFS (REST)** | HTTP requests to the NameNode (port 9870), which redirects to the DataNode | Only needs `requests`; slower for large files |

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow import fs as pafs

# libhdfs needs to find the Hadoop Java libraries
os.environ["CLASSPATH"] = subprocess.run(["hadoop", "classpath", "--glob"],
                                         capture_output=True, text=True, check=True).stdout.strip()
os.environ["ARROW_LIBHDFS_DIR"] = f"{HADOOP_HOME}/lib/native"

hdfs = pafs.HadoopFileSystem("localhost", 9000)

for info in hdfs.get_file_info(pafs.FileSelector("/lab6", recursive=False)):
    print(f"{info.type.name:<10} {info.path}")

# Parquet from HDFS -> Arrow -> Pandas
wc = pq.read_table("/lab6/gold/wordcount", filesystem=hdfs).to_pandas()
print()
print(wc.sort_values("count", ascending=False).head())

In [ ]:
# Pandas -> HDFS: writing a Parquet file through the same filesystem
summary = wc.assign(length=wc["word"].str.len()).groupby("length", as_index=False)["count"].sum()
pq.write_table(pa.Table.from_pandas(summary), "/lab6/gold/count_by_length.parquet", filesystem=hdfs)
sh("hdfs dfs -ls -h /lab6/gold")

In [ ]:
# Alternative without libhdfs: WebHDFS, the HDFS REST API
import io, requests

WEBHDFS = "http://localhost:9870/webhdfs/v1"

listing = requests.get(f"{WEBHDFS}/lab6/raw?op=LISTSTATUS").json()
for st in listing["FileStatuses"]["FileStatus"]:
    print(f"{st['pathSuffix']:<16} {st['length'] / 1e6:6.1f} MB  block={st['blockSize'] / 2**20:.0f} MB  replicas={st['replication']}")

# OPEN: the NameNode replies with a redirect to the DataNode that has the block (requests follows it)
r = requests.get(f"{WEBHDFS}/lab6/out/wordcount_mr/part-00000?op=OPEN")
pd.read_csv(io.BytesIO(r.content), sep="\t", names=["word", "count"]).nlargest(3, "count")

Look at the `OPEN` redirect: this is the HDFS architecture in action. The client asks the NameNode **where** the data is and reads **directly from the DataNode**.

### 2.5 Exercise

Do this exercise before shutting down HDFS in section 2.6.

In [ ]:
# ── Exercise 1 (requires the HDFS from section 2) ───────────────────────────
# a) Upload the orders.csv generated in the setup to HDFS with 1 MB blocks.
# b) Use fsck to find out how many blocks it has.
# c) Read the file with sc.textFile and with spark.read.csv. How many partitions does each produce? Why?

# Your solution:
sh(f"hdfs dfs -D dfs.blocksize={2**20} -put -f {ORDERS_CSV} /lab6/raw/orders.csv")
sh("hdfs fsck /lab6/raw/orders.csv -files -blocks | grep 'Total blocks'")

print("sc.textFile   :", sc.textFile(f"{HDFS}/lab6/raw/orders.csv").getNumPartitions(), "partitions")
print("spark.read.csv:", spark.read.csv(f"{HDFS}/lab6/raw/orders.csv", header=True).rdd.getNumPartitions(), "partitions")
# textFile uses Hadoop's InputFormat: one partition per block (split).
# spark.read splits by its own criteria (maxPartitionBytes and number of cores), not by blocks.

### 2.6 Shutting down

In [ ]:
sh("hdfs --daemon stop datanode")
sh("hdfs --daemon stop namenode")
sh("jps", check=False)

---
## Summary

| Concept | Hadoop | In Spark |
|----------|--------|----------|
| Distributed files | HDFS: 128 MB blocks, 3 replicas | `hdfs://` via the FileSystem API |
| Metadata | NameNode (memory) | — (catalogs: Hive Metastore, Unity) |
| Processing | MapReduce: map, shuffle and sort, reduce | Stages separated by shuffles |
| Resources | YARN: ResourceManager, containers | `--master yarn`, executors = containers |
| Moving code to the data | Block locality | `Locality Level` in the Spark UI |
| Completed output | `_SUCCESS` marker | The same marker |
| Settings | `core-site.xml`, `hdfs-site.xml` | `spark.hadoop.*` |
| Cloud | S3A, GCS, ABFS connectors | `s3a://`, `gs://`, `abfs://` |

| Command | What for |
|---------|----------|
| `hdfs dfs -ls / -put / -get / -cat / -du / -rm` | Manipulate files |
| `hdfs fsck <path> -files -blocks -locations` | See blocks and replicas |
| `hdfs dfsadmin -report` | Cluster status |
| `hadoop jar hadoop-streaming.jar -mapper ... -reducer ...` | MapReduce with scripts |
| `spark-submit --master yarn --deploy-mode cluster` | Run Spark on a Hadoop cluster |

In [ ]:
spark.stop()